In [12]:
import pandas as pd 
import requests
from bs4 import BeautifulSoup
import requests
import os
import re
from pathlib import Path

In [24]:
import os
from os import listdir
from os.path import isfile, join
mypath = '/Users/emily.christiansen/Documents/GitHub/Film_Transcripts/url csv'
onlyfiles = [f for f in  listdir(mypath) if isfile(join(mypath, f))]
csv_paths = []
for f in onlyfiles: 
    csv_paths.append(mypath + '/' + f)

We have the following files
- path to the url CSVs
- path to the download locations (each matching the name of the URL csv)

In [25]:
csv_df_list = []
for csv in csv_paths:
    df = pd.read_csv(csv)
    csv_df_list.append(df)

In [26]:
# Get the name of the file that we get the CSV from and make a new directory to put the SRT files into
pattern = r'(\d{4})\.csv$'
download_dirs = []

for path in csv_paths:
    match = re.search(pattern, path)
    if match:
        year = match.group(1)
        download_path = '/Users/emily.christiansen/Documents/GitHub/Film_Transcripts/srt files/' + year
    download_dirs.append(download_path)

In [27]:
download_dirs

['/Users/emily.christiansen/Documents/GitHub/Film_Transcripts/srt files/2001',
 '/Users/emily.christiansen/Documents/GitHub/Film_Transcripts/srt files/2000',
 '/Users/emily.christiansen/Documents/GitHub/Film_Transcripts/srt files/2002']

### Want to change the code so that if the srt files for that year already exist we dont need to re download 

In [28]:
# for download_dir in download_dirs:
# for df in csv_df_list:
for df, download_dir in zip(csv_df_list, download_dirs):
    for url in df['download URL'].to_list():
        if "subtitlecat" in url: 
            response = requests.get(url)
            html_content = response.text
            soup = BeautifulSoup(html_content, 'html.parser')
        
            
            download_link = None
            for link in soup.find_all('a', href=True):
                if('id' in list(link.attrs.keys())):
                    if link['id'] == 'download_en':
                        if link['href'].endswith('.srt') or link['href'].endswith('.srt'): # Add other relevant extensions
                            download_link = link['href']
                            # Handle relative URLs by constructing the absolute URL
                            if not download_link.startswith('http'):
                                # This is a basic example; more robust URL joining might be needed
                                download_link = requests.compat.urljoin(url, download_link) 
                            break
            
            if download_link:
                print(f"Found download link: {download_link}")
            else:
                print("Download link not found.")
    
            if download_link:
                default_filename = download_link.split('/')[-1]
                specific_file_path = os.path.join(download_dir, default_filename)
                if not os.path.exists(download_dir):
                    os.makedirs(download_dir, exist_ok=True)
                    print(f"Created directory: {download_dir}")
            
                # 3. Download the file using the specific path
                print(f"Starting download to {specific_file_path}...")
                file_response = requests.get(download_link, stream=True)
                
                try: 
                    file_response.raise_for_status()
                    with open(specific_file_path, 'wb') as f:
                        for chunk in file_response.iter_content(chunk_size=8192):
                            f.write(chunk)
                    print(f"Successfully downloaded file to: {specific_file_path}")
                except requests.exceptions.HTTPError as e:
                    print("HTTP error occurred:", e)

Found download link: https://www.subtitlecat.com/subs/41/harry-potter-and-the-sorcerers-stone-yify-english-en.srt
Starting download to /Users/emily.christiansen/Documents/GitHub/Film_Transcripts/srt files/2001/harry-potter-and-the-sorcerers-stone-yify-english-en.srt...
Successfully downloaded file to: /Users/emily.christiansen/Documents/GitHub/Film_Transcripts/srt files/2001/harry-potter-and-the-sorcerers-stone-yify-english-en.srt
Found download link: https://www.subtitlecat.com/subs/103/Shrek.2001.1080p.BluRay.x264.YIFY-en.srt
Starting download to /Users/emily.christiansen/Documents/GitHub/Film_Transcripts/srt files/2001/Shrek.2001.1080p.BluRay.x264.YIFY-en.srt...
Successfully downloaded file to: /Users/emily.christiansen/Documents/GitHub/Film_Transcripts/srt files/2001/Shrek.2001.1080p.BluRay.x264.YIFY-en.srt
Found download link: https://subtitlecat.com/subs/544/Monsters.Inc.2001.720p.BluRay.x264.YIFY-en.srt
Starting download to /Users/emily.christiansen/Documents/GitHub/Film_Transcr

TypeError: argument of type 'float' is not a container or iterable

In [25]:
try: 
    file_response.raise_for_status()
    with open(specific_file_path, 'wb') as f:
        for chunk in file_response.iter_content(chunk_size=8192):
            f.write(chunk)
    print(f"Successfully downloaded file to: {specific_file_path}")
except requests.exceptions.HTTPError as e:
    print("HTTP error occurred:", e)

HTTP error occurred: 404 Client Error: Not Found for url: https://www.subtitlecat.com/subs/848/The.Majestic.2001.720p.BluRay.x264.VPPV-en.srt


So Above in the Python code is what is necessary to download the SRT file from that specific website, what we need now is to make the google scraper to get the first search URL

In [23]:
import requests

try:
    file_response.raise_for_status()
except requests.exceptions.HTTPError as e:
    print("HTTP error occurred:", e)
except requests.exceptions.RequestException as e:
    print("A request error occurred:", e)

HTTP error occurred: 404 Client Error: Not Found for url: https://www.subtitlecat.com/subs/848/The.Majestic.2001.720p.BluRay.x264.VPPV-en.srt


### Check in with how many files that I have manually downloaded so far

In [14]:
import os
mypath = '/Users/emily.christiansen/Documents/GitHub/Film_Transcripts/srt files'

In [15]:


from os import listdir
from os.path import isfile, join
onlyfiles = [f for f in listdir(mypath) if isfile(join(mypath, f))]


In [6]:
from bs4 import BeautifulSoup
# Parse the HTML content
soup = BeautifulSoup(response.text, 'html.parser')
# Find the table containing the CO2 emissions data
table = soup.find('table')

In [7]:
from bs4 import BeautifulSoup
import requests

search = 'the batman subtitles'
url = 'https://www.google.com/'

In [8]:

headers = {
	'Accept' : '*/*',
	'Accept-Language': 'en-US,en;q=0.5',
	'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/135.0.0.0 Safari/537.36',
}
parameters = {'q': search}

In [9]:
content = requests.get(url, headers = headers, params = parameters).text
soup = BeautifulSoup(content, 'html.parser')

In [10]:
from googlesearch import search

query = "Today's weather"

# Perform the Google search and fetch the top result
search_results = list(search(query, num_results=1, stop=1, pause=2)) 

# Print the top search result URL
if search_results:
    print("Top search result:", search_results[0])
else:
    print("No results found.")

TypeError: search() got an unexpected keyword argument 'stop'

In [114]:
search('Google').

TypeError: 'generator' object is not subscriptable

In [105]:
for i in soup.find_all():
    for key in list(i.attrs.keys()):
        #print(i[key])
        if i[key] == 'search':
            print(i)

<form action="/search" autocomplete="off" method="GET" role="search"> <div jsdata="MuIEvd;_;R50wad-hHu7GkPIPhs3E6Ag1" jsmodel="b5W85 vNzKHd"> <div class="A8SBwf" data-alt="false" data-biboe="false" data-hp="true" jsaction="lX6RWd:w3Wsmc;aaFXSd:k0wtTd;ocDSvd:duwfG;XmGRxb:mVw6nb;R6Slyc:F3goue;DkpM0b:d3sQLd;IQOavd:dFyQEf;XzZZPe:jI3wzf;Aghsf:AVsnlb;iHd9U:Q7Cnrc;f5hEHe:G0jgYd;vmxUb:j3bJnb;XBqW7:ihYaWc;UkQk6c:VSb4De;nTzfpf:YPRawb;CudXPd:iu9yrc;R2c5O:LuRugf;qiCkJd:ANdidc;Q3vWPd:FtWxqb;NOg9L:HLgh3;uGoIkd:epUokb;zLdLw:eaGBS;H9muVd:J4e6lb;djyPCf:nMeUJf;hBEIVb:nUZ9le;B4WM0e:QgREKd;acb7qb:H8sTz;rcuQ6b:npT2md" jscontroller="cnjECf" jsdata="LVplcb;_;" jsmodel="kjkykd EPRt9d LM7wx Qlyryb EtCx8b Ip3Erc L97mud   jLgnvd       "><style>.A8SBwf{margin:0 auto;max-width:688px;padding-top:6px;position:relative}.RNNXgb{display:flex;z-index:3;position:relative;min-height:50px;background:#fff;border:1px solid #dadce0;box-shadow:0px 3px 10px 0px rgba(31, 31, 31, 0.08);border-radius:26px;margin:0 auto;}.emcav .RN

In [32]:
search = soup.find(id = 'search')
first_link = search.find('a')

AttributeError: 'NoneType' object has no attribute 'find'

In [ ]:
print(first_link['href'])

In [17]:
# Extract the table headers
headers = [header.text.strip() for header in table.find_all('th')]
# Extract the rows of data
rows = []
for row in table.find_all('tr')[1:]: # Skip the header row
    cells = row.find_all('td')
    row_data = [cell.text.strip() for cell in cells]
    rows.append(row_data)
# Print the headers and first row to check the data
print(headers)
print(rows[0])

['', 'Fossil CO2 emissions (tons)', 'CO2 emissions change', 'CO2 emissions per capita', 'Population', 'Pop. change', "Share of World's CO2 emissions"]
['2022', '4,853,780,240', '1.78%', '14.21', '341,534,046', '0.4%', '12.60%']
